<a href="https://colab.research.google.com/github/iannickgagnon/notebooks/blob/main/design_pattern_decorator_strategy_visitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from __future__ import annotations

from random import randint, choice

from dataclasses import dataclass, asdict

from typing import Protocol, runtime_checkable

# Generic Knapsack object class

In [32]:
@dataclass
class KnapsackObject:
  """
  Generic Knapsack object class.

  Attributes:
    identifier (int): Object identifier.
    weight (int): Weight of the object.
    value (int): Value of the object.
  """
  identifier: int
  weight: int
  value: int

  def __post__init__(self):
    """
    Validate attributes after initialization.
    """
    if self.weight <= 0:
      raise ValueError("Weight must be > 0")

    if self.value < 0:
      raise ValueError("Value cannot be negative")

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Visitor pattern method for double dispatching.
    """
    raise NotImplementedError("The accept method is only implemented in the concrete subclasses.")

# Concrete KnapsckObject classes

In [34]:
class Television(KnapsackObject):
  """
  Concrete KanpsackObject class that corresponds to a television.

  Attributes:
    name (str): Name of the object.
  """
  name:str = "Television"

  def __init__(self, identifier: int, weight: int, value: int):
    """"
    Calls the parent class constructor.
    """
    super().__init__(identifier=identifier, weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_television(self)


class Vase(KnapsackObject):
  """
  Concrete KnapsackObject class that corresponds to a vase.
  """
  name:str  = "Vase"

  def __init__(self, identifier: int, weight: int, value: int):
    """
    Calls the parent class constructor.
    """
    super().__init__(identifier=identifier, weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_vase(self)

# Concrete visitors

In [35]:
@runtime_checkable
class KnapsackObjectVisitor(Protocol):
  """
  Visitor interface for KnapsackObject elements.

  Protocol defines a structural interface, which amounts to duck typing
  with type checking.

  That means:
    - A class does NOT need to inherit from `KnapsackObjectVisitor` to be
      considered a valid visitor.
    - It only needs to provide the same methods with compatible signatures.

  Example:

    class MyVisitor:
      def visit_television(self, obj: Television) -> float: ...
      def visit_vase(self, obj: Vase) -> float: ...

    `MyVisitor` is accepted anywhere a `KnapsackObjectVisitor` is expected,
    because it has the right method names/signatures.

  This is useful for the Visitor pattern because you can add new visitors
  (new operations) without modifying existing object classes, and without
  forcing visitors to inherit from a common base class.

  Normally, `Protocol` is meant for static type checkers (mypy/pyright).

  At runtime, Python does not enforce protocol conformance automatically.

  Decorating the Protocol with @runtime_checkable enables runtime checks like:

      isinstance(visitor, KnapsackObjectVisitor)

  With the decorator:
    - `isinstance` will return True if the object has the required attributes
      (here: `visit_television` and `visit_vase`) with a compatible shape.

    - It’s still not perfect "full type checking" at runtime; it’s mainly an
      structural attribute-presence check, not a guarantee about argument
      and return types.

  Without the decorator:
    - isinstance(visitor, KnapsackObjectVisitor) raises a TypeError.

  The `...` is the literal Python `Ellipsis` object.

  In a Protocol, these method bodies are stubs:
    - They say “this method must exist” and define its signature.
    - They do not provide an implementation.

  The type checker reads these stubs as abstract requirements.

  With this Protocol, the `accept()` methods can be typed as:
      def accept(self, visitor: KnapsackObjectVisitor) -> float:
          ...

  Then any new visitor class (InverseWeightVisitor, ValueDensityVisitor, etc.)
  will type-check as long as it defines the required methods with no inheritance
  required.
  """
  def visit_television(self, obj: Television) -> float: ...
  def visit_vase(self, obj: Vase) -> float: ...

class InverseWeightVisitor:
  """
  Returns the inverse of the object's weight.
  """
  def visit_television(self, obj: Television) -> float:
    return 1.0 / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return 1.0 / obj.weight

class ValueDensityVisitor:
  """
  Returns the value density (value/weight) of the object.
  """
  def visit_television(self, obj: Television) -> float:
    return obj.value / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return obj.value / obj.weight

# Data generator

In [36]:
@dataclass
class DataGenerator:
  """
  Generates a list of random objects for the Knapsack Problem.

  Attributes:
    weight_min_television_lb (int): Minimum weight of a television in pounds (lb).
    weight_max_television_lb (int): Maximum weight of a television in pounds (lb).
    value_min_television (int): Minimum value of a television in dollars.
    value_max_television (int): Maximum value of a television in dollars.
    weight_min_vase_lb (int): Minimum weight of a vase in pounds (lb).
    weight_max_vase_lb (int): Maximum weight of a vase in pounds (lb).
    value_min_vase (int): Minimum value of a vase in dollars.
    velue_max_vase (int): Maximum value of a vase in dollars.
  """
  weight_min_television_lb: int = 50
  weight_max_television_lb: int = 300
  value_min_television: int = 100
  value_max_television: int = 1500
  weight_min_vase_lb: int = 1
  weight_max_vase_lb: int = 10
  value_min_vase: int = 5
  value_max_vase: int = 3000

  def __post__init__(self):
    """
    Ensures that attributes (i.e. weights and values) are greater than zero.
    """
    accumulator: list[str] = []
    for attribute, value in asdict(self).items():
      if value <= 0:
        accumulator.append(attribute)
    if accumulator:
      raise ValueError(f"The following attributes must be greater than zero : {accumulator}")


  def generate(self, nb_objects: int = 10, verbose: bool = True) -> list[KnapsackObject]:
    """
    Generates a list of nb_objects random objects.

    Args:
      nb_objects (int, optional): The number of objects to generate. Defaults to 10.
      verbose (bool, optional): Whether to print progress or not. Defaults to True.

    Returns:
      (tuple[KnapsackObject]): A list of random KnapsackObject instances.

    Raises:
      ValueError: When the supplied number of objects is less than zero.
    """

    # Validate number of objects
    if nb_objects < 0:
      raise ValueError("The number of objects to generate must be >= 0")

    # Initialize return container
    objects: list[KnapsackObject] = []

    # Generate random objects
    for i in range(nb_objects):

      # NOTE: Type is simplified for readability
      obj_factory: tuple = choice([(Television,
                            self.weight_min_television_lb,
                            self.weight_max_television_lb,
                            self.value_min_television,
                            self.value_max_television),
                            (Vase,
                            self.weight_min_vase_lb,
                            self.weight_max_vase_lb,
                            self.value_min_vase,
                            self.value_max_vase)])

      weight_min: int = obj_factory[1]
      weight_max: int = obj_factory[2]
      value_min: int = obj_factory[3]
      value_max: int = obj_factory[4]

      random_weight: int = randint(weight_min, weight_max)
      random_value: int = randint(value_min, value_max)

      random_object: KnapsackObject = obj_factory[0](identifier=i + 1,
                                                     weight=random_weight,
                                                     value=random_value)

      objects.append(random_object)

      # Show progress
      if verbose:
        print(f"Added object no.{i + 1}: {random_object}")

    return objects

# Run generator
objects = DataGenerator().generate()

Added object no.1: Television(identifier=1, weight=145, value=808)
Added object no.2: Television(identifier=2, weight=169, value=825)
Added object no.3: Television(identifier=3, weight=230, value=368)
Added object no.4: Television(identifier=4, weight=207, value=867)
Added object no.5: Vase(identifier=5, weight=7, value=808)
Added object no.6: Television(identifier=6, weight=249, value=1177)
Added object no.7: Vase(identifier=7, weight=7, value=956)
Added object no.8: Vase(identifier=8, weight=4, value=1769)
Added object no.9: Television(identifier=9, weight=135, value=585)
Added object no.10: Television(identifier=10, weight=290, value=1102)


# Do something with the visitors
Calculate the inverse weights of the objects in the list.

In [37]:
inverse_weight_visitor = InverseWeightVisitor()
value_density_visitor = ValueDensityVisitor()

inverse_weights: list[float] = []
value_densities: list[float] = []
for knapsack_obj in objects:
  inverse_weights.append(knapsack_obj.accept(inverse_weight_visitor))
  value_densities.append(knapsack_obj.accept(value_density_visitor))

for weight, density in zip(inverse_weights, value_densities):
  print(f"inverse weight (1/lb) = {weight}\n"
        f"value density  ($/lb) = {density}\n")

inverse weight (1/lb) = 0.006896551724137931
value density  ($/lb) = 5.572413793103448

inverse weight (1/lb) = 0.005917159763313609
value density  ($/lb) = 4.881656804733728

inverse weight (1/lb) = 0.004347826086956522
value density  ($/lb) = 1.6

inverse weight (1/lb) = 0.004830917874396135
value density  ($/lb) = 4.188405797101449

inverse weight (1/lb) = 0.14285714285714285
value density  ($/lb) = 115.42857142857143

inverse weight (1/lb) = 0.004016064257028112
value density  ($/lb) = 4.7269076305220885

inverse weight (1/lb) = 0.14285714285714285
value density  ($/lb) = 136.57142857142858

inverse weight (1/lb) = 0.25
value density  ($/lb) = 442.25

inverse weight (1/lb) = 0.007407407407407408
value density  ($/lb) = 4.333333333333333

inverse weight (1/lb) = 0.0034482758620689655
value density  ($/lb) = 3.8



# Knapsack problem instance

In [43]:
@dataclass
class KnapsackProblem:
  """
  Contains necessary data for the KnapSackProblemSolver.

  Attributes:
    identifiers (list[int]): A list of unique identifiers for the objects.
    weights (list[int]): The weights of the objects.
    values (list[int]): The values of the objects.
    value_densities (list[float]): The value densities (i.e. value / weight) of the objects.
  """
  identifiers: list[int]
  weights: list[int]
  values: list[int]
  value_densities: list[float]

# Knapsack problem parser

In [44]:


class KnapsackProblemParser:
  """
  Generates a KnapsackProblem instance from data.

  Attributes:
    objects (list[KnapsackObject]): Objects to choose from.
    value_densities (list[float]): Order-matching value densities of the objects.
  """

  def __init__(self,
               objects: list[KnapsackObject],
               value_densities: list[float]):

    # Store objects and inverse weights
    self.objects = objects
    self.value_densities = inverse_weights

  def parse(self) -> KnapsackProblem:
    """
    Builds KnapsackProblem object from data.

    Returns:
      (KnapsackProblem): Generated problem instance.
    """

    # Extract identifiers, weights and values as lists
    identifiers: list[int] = [obj.identifier for obj in objects]
    weights: list[int] = [obj.weight for obj in objects]
    values: list[int] = [obj.value for obj in objects]

    # Instanciate problem instance
    knapsack_problem = KnapsackProblem(identifiers=identifiers,
                                       weights=weights,
                                       values=values,
                                       value_densities=self.value_densities)

    return knapsack_problem

# Create KnapsackProblem instance from parser

In [47]:
parser = KnapsackProblemParser(objects=objects,
                               value_densities=value_densities)

problem_instance: KnapsackProblem = parser.parse()

print(problem_instance)

KnapsackProblem(identifiers=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], weights=[145, 169, 230, 207, 7, 249, 7, 4, 135, 290], values=[808, 825, 368, 867, 808, 1177, 956, 1769, 585, 1102], value_densities=[0.006896551724137931, 0.005917159763313609, 0.004347826086956522, 0.004830917874396135, 0.14285714285714285, 0.004016064257028112, 0.14285714285714285, 0.25, 0.007407407407407408, 0.0034482758620689655])
